# Utilities

In [ ]:
!pip install speechbrain

In [ ]:
# --- logger.py Content ---
import torch
from pathlib import Path
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import logging
import sys

def setup_logger(log_dir: Path,console_verbosity:str , log_name: str = "training.log",enable_file = True):
    """
    Initializes and configures a logger for training or workflow monitoring.

    Args:
        log_dir (Path): Directory where the log file will be stored. Created if it doesn't exist.
        console_verbosity (str): Verbosity level for console output. Accepts:
            - "none": disables console logging (sets level above CRITICAL)
            - "all": enables full debug-level logging
            - any other value defaults to INFO-level logging
        log_name (str, optional): Name of the log file to create. Defaults to "training.log".
        enable_file (bool, optional): If True, enables logging to file. Defaults to True.

    Returns:
        logging.Logger: Configured logger instance named "Trainer" with attached handlers.
    """

    log_dir.mkdir(parents=True, exist_ok=True)
    log_path = log_dir / log_name

    logger = logging.getLogger("Trainer")
    logger.setLevel(logging.DEBUG)
    logger.handlers.clear()

    # Always add file handler (UTF-8 for emoji safety)
    if enable_file:
        file_handler = logging.FileHandler(log_path, encoding="utf-8")
        file_handler.setLevel(logging.DEBUG)
        file_fmt = logging.Formatter("%(asctime)s - %(message)s")
        file_handler.setFormatter(file_fmt)
        logger.addHandler(file_handler)

    console_handler = logging.StreamHandler(sys.stdout)
    if console_verbosity == "none":
        console_handler.setLevel(51)
    elif console_verbosity == "all":
        console_handler.setLevel(logging.DEBUG)
    else: console_handler.setLevel(logging.INFO)
    console_fmt = logging.Formatter("%(message)s")
    console_handler.setFormatter(console_fmt)
    logger.addHandler(console_handler)

    return logger


def get_report(
    y_true: torch.Tensor,
    y_pred: torch.Tensor,
    average: str = 'macro',
    target_names: list[str] = None,
):
    """
    Generates and logs/prints a classification report and confusion matrix.
    """
    # Convert lists to tensors if needed
    if isinstance(y_true, list): y_true = torch.tensor(y_true)
    if isinstance(y_pred, list): y_pred = torch.tensor(y_pred)

    if not target_names:
        num_class = len(torch.unique(y_true))
        target_names = [str(i) for i in range(num_class)]

    f1 = f1_score(y_true, y_pred, zero_division=0, average=average)
    report = classification_report(y_true, y_pred, target_names=target_names, zero_division=0)
    cm = confusion_matrix(y_true, y_pred)

    msg = (
        f"\n--- Validation Report ---\n"
        #f"F1 Score ({average}): {f1:.4f}\n\n"
        f"Classification Report:\n{report}\n"
        f"Confusion Matrix:\n{cm}\n"
        f"-------------------------\n"
    )

    return f1, msg

In [ ]:
# --- segment_audio.py Content ---
import os
import librosa
import soundfile as sf
import pandas as pd
from tqdm import tqdm
import numpy as np

def segment_audio_dataset(
    csv_path: str,
    audio_root: str,
    out_dir: str,
    target_sr: int = 16000,
    clip_duration: int = 3,
    padding_mode: str = "silence"
):
    """
    Segments audio files into fixed-length clips and saves them.

    Parameters
    ----------
    csv_path : str
        Path to CSV containing 'filename' and 'target' columns.
    audio_root : str
        Root directory containing audio files organized by label folders.
    out_dir : str
        Output directory to save segmented clips.
    target_sr : int, default=16000
        Target sampling rate.
    clip_duration : int, default=3
        Duration of each clip in seconds.
    padding_mode : str, default="silence"
        How to pad short clips:
        - "silence": pad with zeros (librosa.fix_length).
        - "repeat": tile audio until length is reached (like pad_random).

    Returns
    -------
    pd.DataFrame
        DataFrame containing segmented filenames and labels.
    """
    os.makedirs(out_dir, exist_ok=True)
    clip_samples = target_sr * clip_duration

    df = pd.read_csv(csv_path)
    segmented_rows = []

    for idx, row in tqdm(df.iterrows(), total=len(df)):
        file_name = row["filename"]
        label = row["target"]

        folder_path = os.path.join(audio_root, str(label))
        audio_path = os.path.join(folder_path, file_name)

        # Try alternative extensions if missing
        if not os.path.exists(audio_path):
            base = os.path.splitext(file_name)[0]
            found = False
            for ext in ["wav", "mp3", "m4a"]:
                alt = os.path.join(folder_path, f"{base}.{ext}")
                if os.path.exists(alt):
                    audio_path = alt
                    found = True
                    break
            if not found:
                print("MISSING:", file_name)
                continue

        try:
            audio, sr = librosa.load(audio_path, sr=target_sr, mono=True)
        except Exception as e:
            print("ERROR LOADING FILE:", audio_path, e)
            continue

        total_samples = len(audio)
        num_segments = total_samples // clip_samples

        if num_segments == 0:
            # Handle short clips
            if padding_mode == "silence":
                padded = librosa.util.fix_length(audio, clip_samples)
            elif padding_mode == "repeat":
                num_repeats = int(clip_samples / total_samples) + 1
                padded = np.tile(audio, num_repeats)[:clip_samples]
            else:
                raise ValueError("padding_mode must be 'silence' or 'repeat'")

            out_name = f"{os.path.splitext(file_name)[0]}_seg0.wav"
            out_path = os.path.join(out_dir, out_name)
            sf.write(out_path, padded, target_sr)
            segmented_rows.append([out_name, label])
        else:
            # Regular segmentation
            for seg_id in range(num_segments):
                start = seg_id * clip_samples
                end = start + clip_samples
                clip = audio[start:end]

                out_name = f"{os.path.splitext(file_name)[0]}_seg{seg_id}.wav"
                out_path = os.path.join(out_dir, out_name)
                sf.write(out_path, clip, target_sr)
                segmented_rows.append([out_name, label])

    seg_df = pd.DataFrame(segmented_rows, columns=["filename", "target"])
    seg_df.to_csv(os.path.join(out_dir, "train_audio_segmented.csv"), index=False)

    return seg_df

In [ ]:
# --- dataset.py Content ---
from __future__ import annotations
import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset
import torchaudio
from torch import Tensor
from tqdm import tqdm
import torch.nn as nn
import random

# --- Copy this function from their code ---
def pad_random(x: np.ndarray, max_len: int = 64600):
    """
    Pads or crops a 1D numpy array to a fixed length.
    - If x_len >= max_len: returns a random crop of max_len.
    - If x_len < max_len: tiles (repeats) x until it reaches max_len.
    """
    x_len = x.shape[0]

    # if duration is already long enough
    if x_len >= max_len:
        # Take a random segment
        stt = np.random.randint(x_len - max_len + 1) # +1 to include the last possible start
        return x[stt:stt + max_len]

    # if too short, tile (repeat)
    num_repeats = int(max_len / x_len) + 1
    padded_x = np.tile(x, (num_repeats))[:max_len]
    return padded_x
# --- End of copied function ---


from typing import Any, Tuple

class DeepFakeDataset(Dataset):
    def __init__(self, audio_dir: str, label_file_path: str, label_transform = None):
        self.labels_df = pd.read_csv(label_file_path)
        self.audio_dir = audio_dir
        self.label_transform = label_transform

        # --- Add these ---
        self.target_sample_rate = 16000  # Based on the AASIST model
        self.target_length = 64600       # Based on their pad function

        # --- REMOVED self.resampler from here ---
        # Do not initialize the resampler here!

    def __len__(self) -> int:
        return len(self.labels_df)

    def __getitem__(self, index) -> Tuple[Tensor, Any]:
        # Get filename and label
        file_name = self.labels_df.iloc[index, 0]
        label = self.labels_df.iloc[index, 1]

        # Load audio
        file_path = os.path.join(self.audio_dir, file_name)
        waveform, sample_rate = torchaudio.load(file_path)

        # --- Apply AASIST pre-processing ---

        # 1. Resample to 16kHz if necessary
        if sample_rate != self.target_sample_rate:
            # --- FIX: Create the resampler on-the-fly ---
            resampler = torchaudio.transforms.Resample(
                orig_freq=sample_rate,
                new_freq=self.target_sample_rate
            )
            waveform = resampler(waveform)
            # --- End of fix ---

        # 2. Convert to mono
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0)

        # 3. Squeeze to 1D tensor
        waveform = waveform.squeeze() # Shape: [time]

        # 4. Convert to numpy for padding
        waveform_np = waveform.numpy()

        # 5. Apply their exact padding/cropping logic
        waveform_padded_np = pad_random(waveform_np, self.target_length)

        # 6. Convert back to a float tensor
        final_waveform_tensor = torch.from_numpy(waveform_padded_np).float()

        # --- End of pre-processing ---

        if self.label_transform:
            label = self.label_transform(label)

        final_label = torch.tensor(label).long()

        return final_waveform_tensor, final_label

# --- 1. Custom Transform: Gaussian Noise ---
class AddGaussianNoise(object):
    def __init__(self, min_snr_db=10, max_snr_db=30, p=0.5):
        self.min_snr_db = min_snr_db
        self.max_snr_db = max_snr_db
        self.p = p

    def __call__(self, waveform):
        if random.random() > self.p:
            return waveform

        signal_power = waveform.pow(2).mean()
        if signal_power == 0:
            return waveform

        snr_db = random.uniform(self.min_snr_db, self.max_snr_db)
        snr = 10 ** (snr_db / 10)

        noise_power = signal_power / snr
        noise = torch.randn_like(waveform) * torch.sqrt(noise_power)
        return waveform + noise

# --- 2. Custom Transform: Speed Perturbation ---
class SpeedPerturb(object):
    def __init__(self, sample_rate, p=0.5):
        self.sample_rate = sample_rate
        self.p = p
        # Speed factors: 0.9x (slower/deeper) to 1.1x (faster/higher)
        self.speeds = [0.9, 1.0, 1.1]

    def __call__(self, waveform):
        if random.random() > self.p:
            return waveform

        speed = random.choice(self.speeds)
        if speed == 1.0:
            return waveform

        # Resample acts as speed perturbation (changes duration + pitch)
        new_sr = int(self.sample_rate * speed)
        resampler = torchaudio.transforms.Resample(orig_freq=self.sample_rate, new_freq=new_sr)
        return resampler(waveform)

# --- 3. The Full Robust Dataset ---
class SpeakerDataset(Dataset):
    def __init__(self, csv_path: str, audio_dir: str, label_map: dict = None, eval_mode=False):
        self.df = pd.read_csv(csv_path)
        self.audio_dir = audio_dir
        self.eval_mode = eval_mode
        self.sample_rate = 16000
        self.n_mels = 80
        self.target_len = 48000  # 3 seconds * 16000 Hz

        # --- FEATURE EXTRACTION ---
        self.compute_mfcc = torchaudio.transforms.MFCC(
            sample_rate=self.sample_rate,
            n_mfcc=self.n_mels,
            melkwargs={"n_fft": 400, "win_length": 400, "hop_length": 160, "n_mels": 80}
        )

        # --- AUGMENTATIONS (Waveform Level) ---
        self.add_noise = AddGaussianNoise(min_snr_db=15, max_snr_db=35, p=0.5)
        self.speed_perturb = SpeedPerturb(self.sample_rate, p=0.5)

        # --- AUGMENTATIONS (Spectrogram Level) ---
        self.time_mask = torchaudio.transforms.TimeMasking(time_mask_param=15)
        self.freq_mask = torchaudio.transforms.FrequencyMasking(freq_mask_param=15)

        # --- NORMALIZATION ---
        # Instance Norm 1D (treats 80 features as channels)
        self.instance_norm = nn.InstanceNorm1d(self.n_mels)

        # --- LABELS ---
        self.label_map = label_map
        if not self.eval_mode and self.label_map is None:
            self.unique_labels = sorted(self.df['target'].unique())
            self.label_map = {lbl: i for i, lbl in enumerate(self.unique_labels)}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        filename = row['filename']
        filepath = os.path.join(self.audio_dir, filename)

        # 1. LOAD
        try:
            waveform, sr = torchaudio.load(filepath)
        except:
            # Fallback for bad files
            waveform = torch.zeros(1, self.target_len)
            sr = self.sample_rate

        # 2. RESAMPLE
        if sr != self.sample_rate:
            resampler = torchaudio.transforms.Resample(sr, self.sample_rate)
            waveform = resampler(waveform)

        # 3. AUGMENT WAVEFORM (Train Only)
        if not self.eval_mode:
            # A. Speed Perturb (Changes length!)
            waveform = self.speed_perturb(waveform)
            # B. Add Noise
            waveform = self.add_noise(waveform)

        # 4. FIX LENGTH (Crucial after speed perturb)
        # Pad or Crop to exactly 3 seconds
        curr_len = waveform.shape[1]
        if curr_len < self.target_len:
            pad_amt = self.target_len - curr_len
            waveform = torch.nn.functional.pad(waveform, (0, pad_amt))
        elif curr_len > self.target_len:
            start = random.randint(0, curr_len - self.target_len)
            waveform = waveform[:, start:start+self.target_len]

        # 5. COMPUTE MFCC
        mfcc = self.compute_mfcc(waveform) # [1, 80, 301]

        # 6. INSTANCE NORMALIZATION (Always)
        # Normalize the 80 channels to have mean=0, std=1
        mfcc = self.instance_norm(mfcc)

        # 7. SPEC AUGMENT (Train Only)
        if not self.eval_mode:
            mfcc = self.time_mask(mfcc)
            mfcc = self.freq_mask(mfcc)

        mfcc = mfcc.squeeze(0) # Final shape: [80, 301]

        if self.eval_mode:
            return mfcc, filename
        else:
            label_str = row['target']
            label_idx = self.label_map[label_str]
            return mfcc, torch.tensor(label_idx, dtype=torch.long)

# Audio Splitting and Augmentation Pipeline
The following cells handle the data preparation: segmenting the audio and generating augmentations.

In [ ]:
DATASET_ROOT = r"/kaggle/input/comsys-hackathon-6-task-C-3-0-semi-supervised-10/Comsys Task C dataset/Comsys Task C dataset"
original_CSV_path = r"/kaggle/input/comsys-hackathon-6-task-C-3-0-semi-supervised-10/train_audio.csv"
NEW_OUTPUT_DIR = r"/kaggle/working/segmented_train"

In [ ]:
seg_df = segment_audio_dataset(csv_path=original_CSV_path,
    audio_root=DATASET_ROOT,
    out_dir=NEW_OUTPUT_DIR,
    padding_mode="repeat"
)

print(seg_df.head())

  4%|▍         | 4/94 [00:00<00:18,  4.95it/s]/tmp/ipykernel_88/3964527112.py:70: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, sr = librosa.load(audio_path, sr=target_sr, mono=True)
/usr/local/lib/python3.11/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
  7%|▋         | 7/94 [00:01<00:09,  8.76it/s]

ERROR LOADING FILE: /kaggle/input/comsys-hackathon-6-task-C-3-0-semi-supervised-10/Comsys Task C dataset/Comsys Task C dataset/10/audio_6.mp3 


 16%|█▌        | 15/94 [00:02<00:10,  7.51it/s]Note: Illegal Audio-MPEG-Header 0x36372c33 at offset 3466560.
Note: Trying to resync...
Note: Skipped 1024 bytes in input.
[src/libmpg123/parse.c:wetwork():1349] error: Giving up resync after 1024 bytes - your stream is not nice... (maybe increasing resync limit could help).
 20%|██        | 19/94 [00:03<00:17,  4.24it/s]Note: Illegal Audio-MPEG-Header 0x362c3232 at offset 2952960.
Note: Trying to resync...
Note: Skipped 1024 bytes in input.
[src/libmpg123/parse.c:wetwork():1349] error: Giving up resync after 1024 bytes - your stream is not nice... (maybe increasing resync limit could help).
/tmp/ipykernel_88/3964527112.py:70: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, sr = librosa.load(audio_path, sr=target_sr, mono=True)
/usr/local/lib/python3.11/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa vers

           filename  target
0  audio_1_seg0.wav       1
1  audio_1_seg1.wav       1
2  audio_1_seg2.wav       1
3  audio_1_seg3.wav       1
4  audio_1_seg4.wav       1


In [ ]:
import shutil
shutil.move(r"/kaggle/working/segmented_train/train_audio_segmented.csv",r"/kaggle/working")

'/kaggle/working/train_audio_segmented.csv'

In [ ]:
import pandas as pd
import os
import torch
import torchaudio
import torchaudio.functional as F
import torchaudio.transforms as T
import random
import numpy as np
from tqdm import tqdm

# --- CONFIG ---
INPUT_CSV = r"/kaggle/working/train_audio_segmented.csv"
INPUT_ROOT = r"/kaggle/working/segmented_train"
OUTPUT_DIR = r"/kaggle/working/augmented_dataset"
OUTPUT_CSV = r"/kaggle/working/train_augmented.csv" # Renamed to part1

# SLICING CONFIG (As requested)
START_IDX = 0
END_IDX = 2144

os.makedirs(OUTPUT_DIR, exist_ok=True)
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"🚀 Processing on device: {DEVICE}")

# --- GPU OPTIMIZED PROCESSOR CLASS ---
class AugmentationProcessor:
    def __init__(self, device):
        self.device = device
        self.sr = 16000

        # 1. Pre-initialize Resamplers (Expensive to create, fast to run)
        # We create one for each speed factor we intend to use
        print("Initializing GPU Transforms...")
        self.resample_09 = T.Resample(self.sr, int(self.sr * 0.9)).to(device)
        self.resample_11 = T.Resample(self.sr, int(self.sr * 1.1)).to(device)
        self.resample_085 = T.Resample(self.sr, int(self.sr * 0.85)).to(device)

        # Codec Sim (16k -> 8k -> 16k)
        self.resample_down_8k = T.Resample(self.sr, 8000).to(device)
        self.resample_up_8k = T.Resample(8000, self.sr).to(device)

        # MuLaw
        self.mulaw_enc = T.MuLawEncoding(256).to(device)
        self.mulaw_dec = T.MuLawDecoding(256).to(device)

    def add_noise(self, waveform, snr_db):
        # Waveform already on GPU
        signal_power = waveform.pow(2).mean()
        if signal_power == 0: return waveform
        noise_power = signal_power / (10 ** (snr_db / 10))
        noise = torch.randn_like(waveform) * torch.sqrt(noise_power)
        return waveform + noise

    def change_pitch(self, waveform, steps):
        # F.pitch_shift supports GPU
        return F.pitch_shift(waveform, self.sr, n_steps=steps)

    def add_reverb(self, waveform, room_scale=1.0):
        # Generate RIR directly on GPU
        rir_len = int(self.sr * 0.1 * room_scale)
        t = torch.linspace(0, 1, rir_len, device=self.device)
        decay = torch.exp(-5 * t)
        noise = torch.randn(rir_len, device=self.device)
        rir = noise * decay
        rir = rir / torch.sum(torch.abs(rir))
        rir = rir.unsqueeze(0) # [1, L]

        # FFT Convolution on GPU
        # Pad to N + M - 1
        n_fft = waveform.shape[-1] + rir.shape[-1] - 1
        sig_f = torch.fft.rfft(waveform, n=n_fft)
        rir_f = torch.fft.rfft(rir, n=n_fft)
        aug = torch.fft.irfft(sig_f * rir_f, n=n_fft)

        return aug[:, :waveform.shape[-1]]

    def fix_length(self, waveform, target_len=48000):
        if waveform.shape[1] < target_len:
            return torch.nn.functional.pad(waveform, (0, target_len - waveform.shape[1]))
        else:
            return waveform[:, :target_len]

# --- MAIN LOOP ---
processor = AugmentationProcessor(DEVICE)

# Load and Slice Data
df = pd.read_csv(INPUT_CSV)
print(f"Total files: {len(df)}")
df_subset = df.iloc[START_IDX:END_IDX]
print(f"Processing slice: {START_IDX} to {END_IDX} ({len(df_subset)} files)")

new_rows = []

for idx, row in tqdm(df_subset.iterrows(), total=len(df_subset)):
    orig_filename = row['filename']
    label = row['target']

    # 1. Load to CPU then move to GPU
    path = os.path.join(INPUT_ROOT, orig_filename)
    try:
        wav, sr = torchaudio.load(path)
    except:
        continue

    wav = wav.to(DEVICE)

    # Resample if needed
    if sr != 16000:
        resampler = T.Resample(sr, 16000).to(DEVICE)
        wav = resampler(wav)

    wav = processor.fix_length(wav) # Baseline 3s

    # 2. Generate Augmentations (All operations on GPU)
    # Using the pre-initialized transforms from 'processor'
    augmentations = {
        "v0_orig": wav,

        # Speed (Using Cached Resamplers)
        "v1_slow": processor.resample_09(wav),
        "v2_fast": processor.resample_11(wav),

        # Pitch
        "v3_pitch_up": processor.change_pitch(wav, 2),
        "v4_pitch_down": processor.change_pitch(wav, -2),

        # Noise
        "v5_noise_quiet": processor.add_noise(wav, 25),
        "v6_noise_loud": processor.add_noise(wav, 10),

        # Reverb
        "v7_reverb_small": processor.add_reverb(wav, 0.5),
        "v8_reverb_large": processor.add_reverb(wav, 1.5),

        # Combos
        "v9_combo_slow_noise": processor.add_noise(processor.resample_09(wav), 20),
        "v10_combo_fast_reverb": processor.add_reverb(processor.resample_11(wav), 1.0),
        "v11_combo_pitch_noise": processor.add_noise(processor.change_pitch(wav, 2), 15),

        # Deepfake Sims
        "v12_codec_8k": processor.resample_up_8k(processor.resample_down_8k(wav)),
        "v13_robotic": processor.mulaw_dec(processor.mulaw_enc(wav)),

        # The Boss Fight
        "v14_hard": processor.add_noise(
            processor.add_reverb(
                processor.resample_085(wav), 1.0
            ), 10
        )
    }

    # 3. Save (Move back to CPU for I/O)
    base_name = os.path.splitext(orig_filename)[0]

    for suffix, aug_wav in augmentations.items():
        aug_wav = processor.fix_length(aug_wav)
        aug_wav_cpu = aug_wav.cpu() # Move to CPU for saving

        new_filename = f"{base_name}_{suffix}.wav"
        save_path = os.path.join(OUTPUT_DIR, new_filename)

        torchaudio.save(save_path, aug_wav_cpu, 16000)

        new_rows.append({
            "filename": new_filename,
            "target": label
        })

# Save Partial CSV
final_df = pd.DataFrame(new_rows)
final_df.to_csv(OUTPUT_CSV, index=False)
print(f"✅ Chunk Done! Created {len(final_df)} samples.")
print(f"CSV Saved to: {OUTPUT_CSV}")

🚀 Processing on device: cuda:0
Initializing GPU Transforms...
Total files: 2144
Processing slice: 0 to 2144 (2144 files)


100%|██████████| 2144/2144 [14:00<00:00,  2.55it/s]

✅ Chunk Done! Created 32160 samples.
CSV Saved to: /kaggle/working/train_augmented.csv


# Semi-Supervised Training Loop

# IMPORTS

In [ ]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
import torch
import torch.nn as nn
from speechbrain.lobes.models.ECAPA_TDNN import ECAPA_TDNN
import math
import torch.nn.functional as F
from tqdm import tqdm
import torch.optim as optim
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
import torchaudio

In [ ]:
import torchaudio.transforms as T

# Setup DataLoader and Device

In [ ]:
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device:{DEVICE}")

Using device:cuda:0


In [ ]:
# Paths
AUGMENTED_DIR = r"/kaggle/working/augmented_dataset"
AUGMENTED_CSV = r"/kaggle/working/train_augmented.csv"

# Directories for artifacts
WORK_DIR = Path("/kaggle/working")
CSV_STORE = WORK_DIR / "training_csv"
MODEL_STORE = WORK_DIR / "best_model_each_hypothesis"
CSV_STORE.mkdir(exist_ok=True)
MODEL_STORE.mkdir(exist_ok=True)

In [ ]:
BATCH_SIZE = 32
LR = 0.001
EPOCHS = 20
CONFIDENCE_THRESHOLD = 0.95
MAX_HYPOTHESES = 3

# Logger
logger = setup_logger(WORK_DIR, console_verbosity="all", log_name="semi_supervised_loop.log")

# --- 2. DATA PREPARATION ---
# Load full data
full_df = pd.read_csv(AUGMENTED_CSV)
logger.info(f"Loaded Full Augmented Dataset: {len(full_df)} samples")

# Create Master Label Map (Critical for consistency across hypotheses)
unique_labels = sorted(full_df['target'].unique())
MASTER_LABEL_MAP = {lbl: i for i, lbl in enumerate(unique_labels)}
REVERSE_LABEL_MAP = {v: k for k, v in MASTER_LABEL_MAP.items()}
NUM_SPEAKERS = len(unique_labels)

Loaded Full Augmented Dataset: 32160 samples


In [ ]:
class SimpleRobustDataset(Dataset):
    def __init__(self, df, audio_dir, label_map, eval_mode=False):
        self.df = df
        self.audio_dir = audio_dir
        self.label_map = label_map
        self.eval_mode = eval_mode
        self.target_len = 48000 # 3 seconds
        self.sample_rate = 16000

        self.compute_mfcc = torchaudio.transforms.MFCC(
            sample_rate=16000, n_mfcc=80,
            melkwargs={"n_fft": 400, "win_length": 400, "hop_length": 160, "n_mels": 80}
        )

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = os.path.join(self.audio_dir, row['filename'])

        try:
            wav, sr = torchaudio.load(path)
        except:
            wav = torch.zeros(1, self.target_len)
            sr = 16000

        if sr != 16000: wav = torchaudio.transforms.Resample(sr, 16000)(wav)

        # Simple fix length (Augmented files should already be 3s, but safety first)
        if wav.shape[1] < self.target_len:
            wav = F.pad(wav, (0, self.target_len - wav.shape[1]))
        else:
            wav = wav[:, :self.target_len]

        mfcc = self.compute_mfcc(wav).squeeze(0) # [80, 301]

        # Label
        if self.eval_mode:
            return mfcc, row['filename']
        else:
            label_idx = self.label_map[row['target']]
            return mfcc, torch.tensor(label_idx, dtype=torch.long)

In [ ]:
class ECAPA_Classifier(nn.Module):
    def __init__(self, num_speakers, C=512):
        super().__init__()
        self.spec_norm = nn.InstanceNorm1d(80)
        self.backbone = ECAPA_TDNN(
            input_size=80, channels=[C, C, C, C, 3*C],
            kernel_sizes=[5, 3, 3, 3, 1], dilations=[1, 2, 3, 4, 1],
            attention_channels=128, lin_neurons=192
        )
        self.embedding_dim = 192

    def forward(self, x):
        x = self.spec_norm(x)
        x = x.permute(0, 2, 1)
        embeddings = self.backbone(x)
        embeddings = embeddings.squeeze(1)
        return embeddings

In [ ]:
class AAMSoftmax(nn.Module):
    def __init__(self, in_features, out_features, scale=30.0, margin=0.2):
        super(AAMSoftmax, self).__init__()
        self.scale = scale
        self.margin = margin
        self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)

    def forward(self, input, label):
        cosine = F.linear(F.normalize(input), F.normalize(self.weight))
        phi = cosine - self.margin
        one_hot = torch.zeros(cosine.size(), device=input.device)
        one_hot.scatter_(1, label.view(-1, 1).long(), 1)
        output = (one_hot * phi) + ((1.0 - one_hot) * cosine)
        output *= self.scale
        return output

In [ ]:
def train_hypothesis_phase(hypothesis_idx, train_df, val_df):
    """
    Trains a model from scratch for a specific hypothesis.
    """
    # Create directory for this hypothesis checkpoints
    hyp_dir = WORK_DIR / f"H_{hypothesis_idx}"
    hyp_dir.mkdir(exist_ok=True)

    # Loaders
    train_ds = SimpleRobustDataset(train_df, AUGMENTED_DIR, MASTER_LABEL_MAP)
    val_ds = SimpleRobustDataset(val_df, AUGMENTED_DIR, MASTER_LABEL_MAP)
    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=4,
        drop_last=True
    )
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

    # Initialize Model & Loss (Fresh Start)
    model = ECAPA_Classifier(num_speakers=NUM_SPEAKERS).to(DEVICE)
    margin_layer = AAMSoftmax(192, NUM_SPEAKERS).to(DEVICE)

    model_param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total trainable parameters in the model: {model_param_count}")

    criterion_param_count = sum(p.numel() for p in margin_layer.parameters() if p.requires_grad)
    print(f"Total trainable parameters in the criterion (AAMSoftmax): {criterion_param_count}")

    total_param_count = model_param_count + criterion_param_count
    print(f"Total trainable parameters (model + criterion): {total_param_count}")

    ce_loss = nn.CrossEntropyLoss().to(DEVICE)
    optimizer = optim.Adam(list(model.parameters()) + list(margin_layer.parameters()), lr=LR, weight_decay=2e-5)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

    best_val_f1 = 0.0

    logger.info(f"--- STARTING HYPOTHESIS {hypothesis_idx} TRAINING ---")
    logger.info(f"Train Size: {len(train_df)} | Val Size: {len(val_df)}")

    for epoch in range(EPOCHS):
        # Train
        model.train()
        run_loss = 0.0
        all_preds, all_labels = [], []

        pbar = tqdm(train_loader, desc=f"H{hypothesis_idx} Ep {epoch+1}/{EPOCHS}")
        for feats, labels in pbar:
            feats, labels = feats.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            embeds = model(feats)
            logits = margin_layer(embeds, labels)
            loss = ce_loss(logits, labels)
            loss.backward()
            optimizer.step()
            run_loss += loss.item()

            # Metrics
            with torch.no_grad():
                cosine = F.linear(F.normalize(embeds), F.normalize(margin_layer.weight))
                preds = cosine.argmax(1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        scheduler.step()

        # Validation
        model.eval()
        val_preds, val_labels = [], []
        with torch.no_grad():
            for feats, labels in val_loader:
                feats, labels = feats.to(DEVICE), labels.to(DEVICE)
                embeds = model(feats)
                cosine = F.linear(F.normalize(embeds), F.normalize(margin_layer.weight))
                val_preds.extend(cosine.argmax(1).cpu().numpy())
                val_labels.extend(labels.cpu().numpy())

        # Reporting
        val_f1, _ = get_report(val_labels, val_preds, average="macro")
        train_f1, _ = get_report(all_labels, all_preds, average="macro")

        logger.info(f"H{hypothesis_idx} Ep {epoch+1} | Train Loss: {run_loss/len(train_loader):.4f} | Train F1: {train_f1:.4f} | Val F1: {val_f1:.4f}")

        # Checkpointing
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            # Save to global best folder
            torch.save(model.state_dict(), MODEL_STORE / f"H{hypothesis_idx}_best_model.pth")
            torch.save(margin_layer.state_dict(), MODEL_STORE / f"H{hypothesis_idx}_best_criterion.pth")
            logger.info(f"🔥 Best H{hypothesis_idx} Model Saved! F1: {best_val_f1:.4f}")

        # Regular Checkpoint inside Hypothesis folder
        if (epoch + 1) % 2 == 0:
            torch.save(model.state_dict(), hyp_dir / f"model_ep{epoch+1}.pth")
            torch.save(margin_layer.state_dict(), hyp_dir / f"criterion_ep{epoch+1}.pth")

    return best_val_f1

In [ ]:
def generate_pseudo_labels(hypothesis_idx, unlabeled_df):
    """
    Runs inference on unlabeled data and returns High-Confidence samples.
    """
    logger.info(f"--- GENERATING PSEUDO-LABELS (H{hypothesis_idx}) ---")

    # Load Best Model for this Hypothesis
    model_path = MODEL_STORE / f"H{hypothesis_idx}_best_model.pth"
    crit_path = MODEL_STORE / f"H{hypothesis_idx}_best_criterion.pth"

    model = ECAPA_Classifier(NUM_SPEAKERS).to(DEVICE)
    margin_layer = AAMSoftmax(192, NUM_SPEAKERS).to(DEVICE)

    model.load_state_dict(torch.load(model_path, map_location=DEVICE))
    margin_layer.load_state_dict(torch.load(crit_path, map_location=DEVICE))
    model.eval()

    # Dataset (Eval Mode)
    ds = SimpleRobustDataset(unlabeled_df, AUGMENTED_DIR, MASTER_LABEL_MAP, eval_mode=True)
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

    new_samples = []

    with torch.no_grad():
        for feats, filenames in tqdm(loader, desc="Pseudo-Labeling"):
            feats = feats.to(DEVICE)
            embeds = model(feats)

            # Calculate Cosine Similarity -> Softmax Probability
            cosine = F.linear(F.normalize(embeds), F.normalize(margin_layer.weight))
            # Scale cosine by 30 (ArcFace scale) to get sharp probs
            probs = F.softmax(cosine * 30.0, dim=1)

            max_probs, preds = torch.max(probs, dim=1)

            for i in range(len(filenames)):
                if max_probs[i].item() > CONFIDENCE_THRESHOLD:
                    pred_label = REVERSE_LABEL_MAP[preds[i].item()]
                    new_samples.append({
                        "filename": filenames[i],
                        "target": pred_label
                    })

    logger.info(f"Found {len(new_samples)} high-confidence samples out of {len(unlabeled_df)}")
    return pd.DataFrame(new_samples)

In [ ]:
# A. Initial Split (Stratified 10%)
labeled_df, unlabeled_pool_df = train_test_split(
    full_df, test_size=0.90, stratify=full_df['target'], random_state=42
)

# Further split labeled into Train/Val for monitoring
train_df, val_df = train_test_split(
    labeled_df, test_size=0.15, stratify=labeled_df['target'], random_state=42
)

logger.info("Initialization Complete.")
logger.info(f"Initial Labeled Pool: {len(labeled_df)} (Train: {len(train_df)}, Val: {len(val_df)})")
logger.info(f"Initial Unlabeled Pool: {len(unlabeled_pool_df)}")

HYPOTHESIS_INDEX = 0

while HYPOTHESIS_INDEX < MAX_HYPOTHESES:

    # 1. Train Hypothesis Model
    best_f1 = train_hypothesis_phase(HYPOTHESIS_INDEX, train_df, val_df)

    # 2. Check Stopping Conditions
    if best_f1 > 0.99: # Stricter condition since we are on robust data
        logger.info(f"🚀 Convergence Reached! F1 {best_f1} > 0.99. Stopping Loop.")
        break

    if len(unlabeled_pool_df) < 100:
        logger.info("⚠️ Unlabeled pool exhausted. Stopping Loop.")
        break

    # 3. Generate Pseudo-Labels
    pseudo_labels_df = generate_pseudo_labels(HYPOTHESIS_INDEX, unlabeled_pool_df)

    if len(pseudo_labels_df) == 0:
        logger.info("⚠️ No new confident samples found. Stopping Loop.")
        break

    # 4. Merge Data (The M-Step)
    # Add pseudo-labels to training set
    train_df = pd.concat([train_df, pseudo_labels_df], ignore_index=True)

    # Remove these samples from the unlabeled pool
    # (Filter out filenames that are now in train_df)
    used_filenames = set(train_df['filename'])
    unlabeled_pool_df = unlabeled_pool_df[~unlabeled_pool_df['filename'].isin(used_filenames)]

    # Save the new training CSV for record
    csv_path = CSV_STORE / f"train_H{HYPOTHESIS_INDEX+1}.csv"
    train_df.to_csv(csv_path, index=False)

    logger.info(f"📈 Hypothesis {HYPOTHESIS_INDEX} Complete.")
    logger.info(f"New Training Size: {len(train_df)}")
    logger.info(f"Remaining Unlabeled: {len(unlabeled_pool_df)}")

    HYPOTHESIS_INDEX += 1

logger.info("✅ Semi-Supervised Learning Loop Finished.")

Initialization Complete.
Initial Labeled Pool: 3216 (Train: 2733, Val: 483)


Initial Unlabeled Pool: 28944
--- STARTING HYPOTHESIS 0 TRAINING ---
Train Size: 2733 | Val Size: 483


H0 Ep 1/2: 100%|██████████| 85/85 [00:14<00:00,  5.67it/s]


H0 Ep 1 | Train Loss: 5.3685 | Train F1: 0.6130 | Val F1: 0.7262
🔥 Best H0 Model Saved! F1: 0.7262


H0 Ep 2/2: 100%|██████████| 85/85 [00:15<00:00,  5.65it/s]


...

H2 Ep 1 | Train Loss: 1.7387 | Train F1: 0.8871 | Val F1: 0.8863
🔥 Best H2 Model Saved! F1: 0.8863


H2 Ep 2/2: 100%|██████████| 924/924 [02:36<00:00,  5.91it/s]


H2 Ep 2 | Train Loss: 0.7184 | Train F1: 0.9590 | Val F1: 0.8844
--- GENERATING PSEUDO-LABELS (H2) ---


Pseudo-Labeling: 100%|██████████| 66/66 [00:10<00:00,  6.56it/s]

Found 1149 high-confidence samples out of 2084
📈 Hypothesis 2 Complete.
New Training Size: 30742
Remaining Unlabeled: 935
✅ Semi-Supervised Learning Loop Finished.


In [ ]:
# --- CONFIGURATION ---
TEST_AUDIO_DIR = r"/kaggle/input/comsys-hackathon-6-task-C-3-0-semi-supervised-10/NewUpdatedTest_All_task2"
OUTPUT_CSV = "submission_tta_10_semisupervised10.csv"

MODEL_PATH = r"/kaggle/working/best_model_each_hypothesis/H2_best_model.pth"
CRITERION_PATH = r"/kaggle/working/best_model_each_hypothesis/H2_best_criterion.pth"

BATCH_SIZE = 1 # Keep 1 for safety with raw audio lengths
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:

# --- 1. REBUILD LABEL MAP ---
# We need to map the predicted ID (0..31) back to the Speaker Name
train_csv_path = r"/kaggle/working/segmented_train/train_audio_segmented.csv"
if not os.path.exists(train_csv_path):
    # Fallback if segmented csv is missing: try the augmented one or manually check
    train_csv_path = r"/kaggle/working/train_audio_segmented.csv"

print(f"Loading label map from: {train_csv_path}")
df_train = pd.read_csv(train_csv_path)
unique_labels = sorted(df_train['target'].unique())
REVERSE_MAP = {i: lbl for i, lbl in enumerate(unique_labels)}
NUM_SPEAKERS = len(unique_labels)
print(f"Loaded {NUM_SPEAKERS} classes.")


Loading label map from: /kaggle/working/train_audio_segmented.csv
Loaded 32 classes.


In [ ]:
class TTA_Engine:
    def __init__(self, sr=16000, device='cpu'):
        self.sr = sr
        self.device = device
        # Cache transforms
        self.resample_09 = T.Resample(sr, int(sr * 0.9)).to(device)
        self.resample_11 = T.Resample(sr, int(sr * 1.1)).to(device)
        self.resample_8k_down = T.Resample(sr, 8000).to(device)
        self.resample_8k_up = T.Resample(8000, sr).to(device)
        self.mfcc = T.MFCC(sample_rate=sr, n_mfcc=80,
                           melkwargs={"n_fft": 400, "win_length": 400, "hop_length": 160, "n_mels": 80}).to(device)

    def get_10_versions(self, waveform):
        versions = []

        # 1. Original
        versions.append(waveform)

        # 2. Speed 0.9 (Downsampled, shorter)
        versions.append(self.resample_09(waveform))

        # 3. Speed 1.1 (Upsampled, longer)
        versions.append(self.resample_11(waveform))

        # 4-5. Pitch
        versions.append(torchaudio.functional.pitch_shift(waveform, self.sr, 2))
        versions.append(torchaudio.functional.pitch_shift(waveform, self.sr, -2))

        # 6. Noise (Baseline)
        noise = torch.randn_like(waveform) * 0.05
        versions.append(waveform + noise)

        # 7. Codec
        versions.append(self.resample_8k_up(self.resample_8k_down(waveform)))

        # 8-9. Reverb Sim
        delay = int(0.05 * self.sr)
        echo = torch.cat([torch.zeros(1, delay, device=self.device), waveform], dim=1)[:, :waveform.shape[1]]
        versions.append(waveform + 0.3 * echo)

        delay_l = int(0.2 * self.sr)
        echo_l = torch.cat([torch.zeros(1, delay_l, device=self.device), waveform], dim=1)[:, :waveform.shape[1]]
        versions.append(waveform + 0.5 * echo_l)

        # 10. Combo (Speed 0.9 + Noise) - THE FIX
        resampled_09 = self.resample_09(waveform)
        # Generate NEW noise matching the RESAMPLED length
        noise_09 = torch.randn_like(resampled_09) * 0.05
        versions.append(resampled_09 + noise_09)

        # PROCESS INTO MFCCs
        batch_mfccs = []
        target_len = 48000

        for wav in versions:
            # Fix Length (Crop or Pad to exactly 3s)
            if wav.shape[1] < target_len:
                wav = F.pad(wav, (0, target_len - wav.shape[1]))
            else:
                wav = wav[:, :target_len]

            batch_mfccs.append(self.mfcc(wav).squeeze(0))

        return torch.stack(batch_mfccs)

In [ ]:
import glob
# --- 4. DATASET ---
class TestDataset(torch.utils.data.Dataset):
    def __init__(self, audio_dir):
        # Recursive glob to ensure we find files even in subfolders
        self.files = glob.glob(os.path.join(audio_dir, "**/*.wav"), recursive=True) + \
                     glob.glob(os.path.join(audio_dir, "**/*.mp3"), recursive=True)

        if len(self.files) == 0:
            raise ValueError(f"❌ No audio files found in {audio_dir}. Check path!")

        print(f"✅ Found {len(self.files)} test files.")

    def __len__(self): return len(self.files)

    def __getitem__(self, idx):
        path = self.files[idx]
        wav, sr = torchaudio.load(path)
        if sr != 16000: wav = torchaudio.transforms.Resample(sr, 16000)(wav)
        return wav, os.path.basename(path)



In [ ]:
# --- 5. EXECUTION ---
print("Initializing Model...")
model = ECAPA_Classifier(num_speakers=NUM_SPEAKERS).to(DEVICE)
criterion = nn.Linear(192, NUM_SPEAKERS, bias=False).to(DEVICE)

model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
crit_state = torch.load(CRITERION_PATH, map_location=DEVICE)
criterion.weight.data = crit_state['weight']
model.eval()

tta_engine = TTA_Engine(device=DEVICE)
test_ds = TestDataset(TEST_AUDIO_DIR)
loader = torch.utils.data.DataLoader(test_ds, batch_size=1, shuffle=False, collate_fn=lambda x: x)

predictions = []
unstable_count = 0

print(f"🚀 Starting TTA Inference on {len(test_ds)} files...")

with torch.no_grad():
    for batch in tqdm(loader):
        waveform, filename = batch[0]
        waveform = waveform.to(DEVICE)

        # 1. TTA Generation
        tta_batch = tta_engine.get_10_versions(waveform)

        # 2. Forward
        embeddings = model(tta_batch)

        # 3. Scoring
        norm_embeds = F.normalize(embeddings, p=2, dim=1)
        norm_weights = F.normalize(criterion.weight, p=2, dim=1)
        cosine_scores = F.linear(norm_embeds, norm_weights)

        # 4. Voting & Consistency Check
        avg_scores = cosine_scores.mean(dim=0)
        final_pred_idx = avg_scores.argmax().item()

        votes = cosine_scores.argmax(dim=1)
        consistency = (votes == final_pred_idx).sum().item() / 10.0

        if consistency < 0.5:
            unstable_count += 1

        predictions.append({
            "filename": filename,
            "target": REVERSE_MAP[final_pred_idx],
            "consistency": consistency
        })

# --- 6. SAVE RESULTS ---
if len(predictions) > 0:
    df_out = pd.DataFrame(predictions)

    # Save Clean Submission
    df_out[["filename", "target"]].to_csv(OUTPUT_CSV, index=False)

    # Save Debug Info
    df_out.to_csv("submission_debug.csv", index=False)

    print(f"\n✅ SUCCESS! Submission saved to: {OUTPUT_CSV}")
    print(f"📊 Stats: {len(df_out)} predictions generated.")
    print(f"⚠️ Unstable Predictions (<50% agreement): {unstable_count} ({unstable_count/len(df_out)*100:.1f}%)")
else:
    print("\n❌ FATAL ERROR: Predictions list is empty. Check the loop.")

Initializing Model...
✅ Found 452 test files.
🚀 Starting TTA Inference on 452 files...


100%|██████████| 452/452 [01:40<00:00,  4.49it/s]


✅ SUCCESS! Submission saved to: submission_tta_10_semisupervised10.csv
📊 Stats: 452 predictions generated.
⚠️ Unstable Predictions (<50% agreement): 129 (28.5%)
